# Companies House MCP & Agent tools tests    

Companies House mcp ran locally and in Cloud Run

Purpose is to 
1. create a Agent tool that connects to Companies HouseMCP Server
2. Retreives and summarises relevant details requested (requires HITL to make the inital request)
3. HITL middleware/interrupts ask questions of the user where there is uncertainty (e.g. the name of the company not an exact match, or not found or address or director details non matching)


In [ ]:
#!export PATH="$HOME/google-cloud-sdk/bin:$PATH"


## MCP - connect to Local service

In [3]:
COMPANIES_HOUSE_API_KEY = os.getenv("COMPANIES_HOUSE_API_KEY")

In [ ]:
# Run MCP server from project root so it can start correctly (avoids "Connection closed")
project_root = Path.cwd()
if not (COMPANIES_HOUSE_API_KEY or os.getenv("COMPANIES_HOUSE_API_KEY")):
    raise ValueError("COMPANIES_HOUSE_API_KEY is not set. Run the cell above and ensure .env in project root contains it.")

# Find npx path - subprocess may not have same PATH as notebook
npx_path = shutil.which("npx") or "/opt/homebrew/bin/npx"

# Ensure PATH includes homebrew bin for subprocess
env = os.environ.copy()
if "/opt/homebrew/bin" not in env.get("PATH", ""):
    env["PATH"] = f"/opt/homebrew/bin:{env.get('PATH', '')}"

client = MultiServerMCPClient(
    {
        "companies-house": {
            "command": npx_path,  # Use full path to npx
            "transport": "stdio",
            "args": ["-y", "companies-house-mcp-server"],
            "env": {
                **env,  # Include PATH in environment
                "COMPANIES_HOUSE_API_KEY": (COMPANIES_HOUSE_API_KEY or "")
            }
        }
    }
)
tools = await client.get_tools()



In [ ]:
# 37 is a lot! Might want to filter down to just the essentials
tools_by_name = [tool.name for tool in tools]
tools_by_name


In [ ]:
agent = create_agent(
    "gpt-4o",
    tools  
)
ch_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "get directors for Goodman's Consulting"}]}
)

In [ ]:
ch_response

## MCP Service running on CloudRun 

In [ ]:
#! pip install google-auth google-auth-oauthlib google-auth-httplib2 requests

In [ ]:
import importlib

import gcp.python_client_iam_mcp

importlib.reload(gcp.python_client_iam_mcp) 

<module 'gcp.python_client_iam_mcp' from '/Users/stevegoodman/dev/fionaa-be/src/gcp/python_client_iam_mcp.py'>

In [ ]:
CH_MCP_SERVICE_URL = os.environ["CH_MCP_SERVICE_URL"]

# Set GOOGLE_APPLICATION_CREDENTIALS to the correct path
from pathlib import Path
import os
import json
credentials_path = Path.cwd().parent / ".secrets" / "fionaa-service-acct.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path.resolve())

client = gcp.python_client_iam_mcp.IAMAuthenticatedMCPClient(CH_MCP_SERVICE_URL)

try:
    # Health check
    print("Checking service health...")
    health = client.health_check()
    print(f"Health: {json.dumps(health, indent=2)}\n")
    
    # List available tools
    print("Fetching available tools...")
    tools_response = client.list_tools()
    tools = tools_response.get("result", {}).get("tools", [])
    print(f"Found {len(tools)} tools\n")
    
    # Show first few tools
    print("Sample tools:")
    for tool in tools[:5]:
        print(f"  - {tool['name']}: {tool['description'][:60]}...")
    
    # Example: Search for companies
    print("\nSearching for companies...")
    search_result = client.call_tool("search_companies", {
        "query": "Goodmans Consulting",
        "items_per_page": 5
    })
    
    if "error" in search_result:
        print(f"Error: {search_result['error']}")
    else:
        result_content = search_result.get("result", {}).get("content", [])
        if result_content:
            print("Search results:")
            print(json.dumps(json.loads(result_content[0]["text"]), indent=2))
    
except Exception as e:
    # Print the full error message (which may contain helpful instructions)
    error_msg = str(e)
    if "401 Unauthorized" in error_msg or "roles/run.invoker" in error_msg:
        # This is a permission error with helpful instructions
        print(error_msg)
    else:
        # Generic error
        print(f"Error: {error_msg}")
    sys.exit(1)

In [4]:
CH_MCP_SERVICE_URL = os.environ["CH_MCP_SERVICE_URL"]

# Set GOOGLE_APPLICATION_CREDENTIALS to the correct path
from pathlib import Path
import os
import json
credentials_path = Path.cwd().parent / ".secrets" / "fionaa-service-acct.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path.resolve())

client = gcp.python_client_iam_mcp.IAMAuthenticatedMCPClient(CH_MCP_SERVICE_URL)

try:
    # Health check
    print("Checking service health...")
    health = client.health_check()
    print(f"Health: {json.dumps(health, indent=2)}\n")
    
    # List available tools
    print("Fetching available tools...")
    tools_response = client.list_tools()
    tools = tools_response.get("result", {}).get("tools", [])
    print(f"Found {len(tools)} tools\n")
    
    # Show first few tools
    print("Sample tools:")
    for tool in tools[:5]:
        print(f"  - {tool['name']}: {tool['description'][:60]}...")
    
    # Example: Search for companies
    print("\nSearching for companies...")
    search_result = client.call_tool("search_companies", {
        "query": "Goodmans Consulting",
        "items_per_page": 5
    })
    
    if "error" in search_result:
        print(f"Error: {search_result['error']}")
    else:
        result_content = search_result.get("result", {}).get("content", [])
        if result_content:
            print("Search results:")
            print(json.dumps(json.loads(result_content[0]["text"]), indent=2))
    
except Exception as e:
    # Print the full error message (which may contain helpful instructions)
    error_msg = str(e)
    if "401 Unauthorized" in error_msg or "roles/run.invoker" in error_msg:
        # This is a permission error with helpful instructions
        print(error_msg)
    else:
        # Generic error
        print(f"Error: {error_msg}")
    sys.exit(1)

Checking service health...
Health: {
  "status": "ok",
  "service": "companies-house-mcp"
}

Fetching available tools...
Found 37 tools

Sample tools:
  - search_companies: Search for companies by name or company number...
  - get_company_profile: Get detailed profile information for a specific company...
  - get_registered_office_address: Get the registered office address of a company...
  - get_registers: Get company registers information...
  - get_insolvency: Get company insolvency information...

Searching for companies...
Search results:
{
  "items": [
    {
      "kind": "searchresults#company",
      "description_identifier": [
        "dissolved-on"
      ],
      "company_status": "dissolved",
      "date_of_creation": "2012-07-11",
      "date_of_cessation": "2017-12-28",
      "company_type": "ltd",
      "company_number": "08139267",
      "address": {
        "address_line_1": "Heskin",
        "locality": "Preston",
        "postal_code": "PR7 5PA",
        "premises": "

# Test Calling the tool from an LLM

In [14]:
import importlib
import gcp.python_client_iam_mcp
importlib.reload(gcp.python_client_iam_mcp)

from pathlib import Path
from langchain_core.tools import StructuredTool
from pydantic import create_model, Field
from typing import Optional

CH_MCP_SERVICE_URL = os.environ["CH_MCP_SERVICE_URL"]

credentials_path = Path.cwd().parent / ".secrets" / "fionaa-service-acct.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path.resolve())

iam_client = gcp.python_client_iam_mcp.IAMAuthenticatedMCPClient(CH_MCP_SERVICE_URL)

# Fetch tool schemas from the server
tool_schemas = iam_client.list_tools().get("result", {}).get("tools", [])
print(f"Found {len(tool_schemas)} tools on server")

_PY_TYPES = {"string": str, "integer": int, "number": float, "boolean": bool, "array": list, "object": dict}

def _make_langchain_tool(schema: dict) -> StructuredTool:
    name = schema["name"]
    description = schema.get("description", "")
    props = schema.get("inputSchema", {}).get("properties", {})
    required = set(schema.get("inputSchema", {}).get("required", []))

    fields = {}
    for prop_name, prop_schema in props.items():
        py_type = _PY_TYPES.get(prop_schema.get("type", "string"), str)
        field_desc = prop_schema.get("description", "")
        if prop_name in required:
            fields[prop_name] = (py_type, Field(description=field_desc))
        else:
            fields[prop_name] = (Optional[py_type], Field(default=None, description=field_desc))

    ArgsModel = create_model(f"{name}_args", **fields)

    def _call(**kwargs):
        # Strip None values so optional params aren't sent
        args = {k: v for k, v in kwargs.items() if v is not None}
        result = iam_client.call_tool(name, args)
        content = result.get("result", {}).get("content", [])
        return content[0]["text"] if content else str(result)

    return StructuredTool(name=name, description=description, func=_call, args_schema=ArgsModel)

tools = [_make_langchain_tool(s) for s in tool_schemas]
print(f"Created {len(tools)} LangChain tools")
[t.name for t in tools]

Found 37 tools on server
Created 37 LangChain tools


['search_companies',
 'get_company_profile',
 'get_registered_office_address',
 'get_registers',
 'get_insolvency',
 'get_exemptions',
 'get_uk_establishments',
 'advanced_company_search',
 'search_all',
 'search_officers',
 'search_disqualified_officers',
 'alphabetical_search',
 'dissolved_search',
 'get_officers',
 'get_officer_appointment',
 'get_corporate_officer_disqualification',
 'get_natural_officer_disqualification',
 'get_officer_appointments_list',
 'get_filing_history',
 'get_filing_history_item',
 'get_charges',
 'get_charge_details',
 'get_persons_with_significant_control',
 'get_psc_corporate_entity_beneficial_owner',
 'get_psc_corporate_entity',
 'get_psc_individual_beneficial_owner',
 'get_psc_individual',
 'get_psc_individual_verification',
 'get_psc_individual_full_record',
 'get_psc_legal_person_beneficial_owner',
 'get_psc_legal_person',
 'get_psc_statement',
 'get_psc_statements_list',
 'get_psc_super_secure_beneficial_owner',
 'get_psc_super_secure',
 'get_docum

In [15]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="you are a helpful assistant that can answer questions about the companies house"
)
agent.invoke({"messages": [HumanMessage(content="Tell me about the directors of company Goodmans Consulting")]})

{'messages': [HumanMessage(content='Tell me about the directors of company Goodmans Consulting', additional_kwargs={}, response_metadata={}, id='5eb73e8d-b7f7-4224-a053-c8c7412916ee'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 288, 'prompt_tokens': 2179, 'total_tokens': 2467, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DGCQ6n6o07ouT2fbzZ4jx05VJuNLu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cc055-97a8-7790-a080-ec5cfcd200aa-0', tool_calls=[{'name': 'search_companies', 'args': {'query': 'Goodmans Consulting', 'items_per_page': 5}, 'id': 'call_f8BcPzsvbYnWCs8UQ46jHEb0', 'type': 'tool_call'}], i

In [ ]:
# This is an interesting response - How would we turn this into an interrupt so the humanan can clarify?


In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="you are a helpful assistant that can answer questions about contents of companies house. You have access to a number of tools to retreive company information or director information"
)
agent.invoke({"messages": [HumanMessage(content="Tell me about the directors of company Goodmans Consulting")]})

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

CH_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions about UK Companies House. "
    "When calling search_companies, you must always pass the 'query' argument with the company name or number from the user's request (it is required)."
)
agent = create_agent(
    model="gpt-4o",
    tools=tools,
    system_prompt=CH_SYSTEM_PROMPT,
)
result = agent.invoke(
    {"messages": [HumanMessage(content="Tell me about the status and business activities  of company Goodman's Consulting, including directors with significant control")]},
    config={"recursion_limit": 25},
)
result["messages"][-1].content


In [ ]:
# Uses CH_SYSTEM_PROMPT from cell above; gpt-4o + recursion_limit for full answer
agent = create_agent(
    model="gpt-4o",
    tools=tools,
    system_prompt=CH_SYSTEM_PROMPT,
)
result = agent.invoke(
    {"messages": [HumanMessage(content="Tell me about the status and business activities  of company Goodmans Consulting, including directors with significant control")]},
    config={"recursion_limit": 25},
)
result["messages"][-1]

### Why "nothing" or empty response?

If you see only 2 messages (Human + AI with `tool_calls`) and the AI `content` is empty:

1. **Empty tool args**: GPT-5 / gpt-5-nano sometimes call `search_companies` with **no arguments** (`args: {}`). The MCP tool **requires** a `query` (company name or number). So the first tool call fails with "Required" and you get no useful result unless the agent runs again and retries with `query` set.
2. **Run not completed**: If the agent run stops after the first LLM step (e.g. only one step executed), you never get the ToolMessage or the final answer. Use full `agent.invoke(...)` and read the **last** message: `response["messages"][-1].content`.

**Fixes**: Use **gpt-4o** for this agent (it passes `query` correctly on the first call), or use the system prompt below so that `search_companies` is always called with the `query` argument.

##  Langhain and/or langgraph workflow

HITL langchain middleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [ ]:
config = {"configurable": {"thread_id": "123"}}

agent.invoke({'messages':HumanMessage("read an email for id 1")}, config=config)

In [ ]:
agent.invoke({'messages':HumanMessage("sent an email to steve@hotmail with subject line 'hi'")}, config=config)

In [ ]:
response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

In [ ]:
response

In [ ]:
from langchain_core.tools import tool


def email_response(state: State) -> str:
    """Given an email, return a response to the email.
    """
    prompt = f"""
    You are a helpful assistant that can help with email responses. Please respond to the email.
    The email is as follows:
    {state["email_input"]["body"]}
    """
    
    messages = [llm.invoke(prompt)]
    # Create interrupt that is shown to the user
    request = {
        "action_request": {
            "action": f"Email Assistant: {state['email_input']['body']}",
            "args": {}
        },
        "config": {
            "allow_ignore": True,  
            "allow_respond": True, 
            "allow_edit": False, 
            "allow_accept": True,  
        },
        # Email to show in Agent Inbox
        "description": messages.content
    }

    # Agent Inbox responds with a list of dicts with a single key `type` that can be `accept`, `edit`, `ignore`, or `response`.  
    response = interrupt([request])[0]

    # If user provides feedback, go to response agent and use feedback to respond to email   
    if response["type"] == "response":
        # Add feedback to messages 
        # Used by the response agent
        messages.append({"role": "user",
                        "content": "User wants to reply to the email. Use this feedback to respond: user_input"
                        })
        # Go to response agent
        
        goto = "response_agent"

    # If user ignores email, go to END
    elif response["type"] == "accept":
        messages.append({"role": "user",
                        "content": f"User wants to reply to the email. with {draft_reply}"
                        })
        print("ACCEPTED REGISTERED")
        goto = END
    elif response["type"] == "ignore":
        goto = END
    update = {
     "messages": messages, "goto": goto
    }

    return Command(resume=[{"type": "response", "args": {"update": [draft_reply]}}])

In [ ]:
from langchain_core.messages import HumanMessage


def process_email_input(state: State) -> str:
    """Logic is to extract the email body and the email subject
    and then pass these to the agent.
    """
    author, to, subject, body, email_id = parse_gmail(state["email_input"])
    return {
        "messages": [HumanMessage(content=f"Processing email: {state['email_input']['body']}")]
    }
    
    
s1 = State(StateInput(email_input={"to": "steve@goodman.com", "from": "fiona@goodman.com", "subject": "test", "body": "test", "id": "test"}))   

process_email_input(s1)



In [ ]:
graph = (StateGraph(State, input=StateInput)
    .add_node("process_email_input", process_email_input)
    .add_node("email_response", email_response)
    
    .add_edge(START, "process_email_input")
    .add_edge("process_email_input", "email_response")
    .add_edge("email_response", END)
)



In [ ]:
import uuid

from langgraph.checkpoint.memory import InMemorySaver

email={"to": "steve@goodman.com", "from": "fiona@goodman.com", "subject": "test", "body": "test", "id": "test"}
checkpointer = InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)
thread_id_2 = uuid.uuid4()
thread_config_2 = {"configurable": {"thread_id": thread_id_2}}



#workflow.invoke({"email_input": email}, config=thread_config_2)

for chunk in workflow.stream({"email_input": email}, config=thread_config_2):
    # Inspect interrupt object if present
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

In [ ]:
from langgraph.types import Command

print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']}")
for chunk in workflow.stream(Command(resume=[{"type": "accept"}]), config=thread_config_2):
    # Inspect interrupt object if present
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

In [ ]:
chunk['email_response']['messages'][-1]

In [ ]:
# Connect to Linked-in MCP

# Find npx path - subprocess may not have same PATH as notebook
npx_path = shutil.which("npx") or "/opt/homebrew/bin/npx"

# Ensure PATH includes homebrew bin for subprocess
env = os.environ.copy()
if "/opt/homebrew/bin" not in env.get("PATH", ""):
    env["PATH"] = f"/opt/homebrew/bin:{env.get('PATH', '')}"

client = MultiServerMCPClient(
    {
        "companies-house": {
            "command": "uv",  # Use full path to npx
            "transport": "stdio",
            "args": ["--directory", " ~/dev/linkedin-mcp-server", "run", "-m", "linkedin_scraper"],
            
        }
    }
)
tools = await client.get_tools()